# SQL Interview Questions — Executed, Not Memorized

Every query below runs against a real SQLite database built in the first cell, so every result you see is a *result*, not a claim. The question set is the standard DS/ML screen canon: filters, joins, aggregations, NULL logic, subqueries vs CTEs, and the thinking patterns that separate "knows syntax" from "works with data daily."

**Rehearsal protocol:** write your query on paper first, then compare — and rehearse saying what each clause *does to the row set*.

## Setup — a realistic mini-warehouse

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
cur = conn.cursor()

cur.executescript('''
CREATE TABLE employees (
    emp_id INTEGER PRIMARY KEY, name TEXT, dept_id INTEGER,
    manager_id INTEGER, salary INTEGER, hired DATE, city TEXT);
CREATE TABLE departments (dept_id INTEGER PRIMARY KEY, dept_name TEXT, budget INTEGER);
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY, emp_id INTEGER, amount REAL,
    status TEXT, placed_at DATE);

INSERT INTO departments VALUES
 (1, 'Engineering', 2000000), (2, 'Data Science', 1500000), (3, 'Sales', 900000),
 (4, 'Marketing', 600000), (5, 'Research', 100000);

INSERT INTO employees VALUES
 (1, 'Ada',    1, NULL, 185000, '2019-03-01', 'Bengaluru'),
 (2, 'Bruno',  1, 1,    120000, '2020-07-15', 'Bengaluru'),
 (3, 'Chen',   2, 1,    140000, '2021-01-10', 'Remote'),
 (4, 'Disha',  2, 3,    132000, '2022-05-20', 'Mumbai'),
 (5, 'Emil',   2, 3,    128000, '2023-09-01', 'Remote'),
 (6, 'Farah',  3, 1,     95000, '2021-11-11', 'Delhi'),
 (7, 'Gustav', 3, 6,     78000, '2024-02-01', 'Delhi'),
 (8, 'Hana',   4, 1,     88000, '2022-06-30', 'Pune'),
 (9, 'Ivan',   NULL, 1,  91000, '2023-04-15', 'Remote'),
 (10,'Jaya',   3, 6,     82000, '2020-10-05', 'Mumbai');

INSERT INTO orders VALUES
 (101, 3, 450.00, 'shipped',    '2026-01-05'),
 (102, 3, 120.50, 'shipped',    '2026-01-18'),
 (103, 4, 890.00, 'returned',   '2026-02-02'),
 (104, 4, 310.00, 'shipped',    '2026-02-14'),
 (105, 6,  75.25, 'cancelled',  '2026-03-01'),
 (106, 7, 640.00, 'shipped',    '2026-03-09'),
 (107, 8, 220.00, 'pending',    '2026-03-15'),
 (108, 3, 515.00, 'shipped',    '2026-03-22'),
 (109, 9,  99.99, 'shipped',    '2026-04-04'),
 (110, 10, 780.00, 'shipped',   '2026-04-11');
''')
conn.commit()

def q(sql):
    return pd.read_sql_query(sql, conn)

q("SELECT * FROM employees LIMIT 5")

,emp_id,name,dept_id,manager_id,salary,hired,city
0,1,Ada,1,NaN,185000,2019-03-01,Bengaluru
1,2,Bruno,1,1.0,120000,2020-07-15,Bengaluru
2,3,Chen,2,1.0,140000,2021-01-10,Remote
3,4,Disha,2,3.0,132000,2022-05-20,Mumbai
4,5,Emil,2,3.0,128000,2023-09-01,Remote


## Q1 — Filter Basics: WHERE Between Dates, IN, NOT IN's NULL Trap

In [2]:
result = q('''
SELECT name, city, salary
FROM employees
WHERE hired BETWEEN '2021-01-01' AND '2022-12-31'
  AND city IN ('Remote', 'Mumbai', 'Delhi')
ORDER BY salary DESC
''')
result

,name,city,salary
0,Chen,Remote,140000
1,Disha,Mumbai,132000
2,Farah,Delhi,95000


**Say out loud:** `BETWEEN` is **inclusive** on both ends. And the trap interviewers probe: **`NOT IN (subquery containing NULL) returns zero rows** — because `x NOT IN (1, NULL)` evaluates to `NULL`, never TRUE. The safe forms are `NOT EXISTS` or `LEFT JOIN ... WHERE right IS NULL`.

## Q2 — GROUP BY + HAVING: departments with more than 2 employees

In [3]:
q('''
SELECT d.dept_name, COUNT(*) AS n_emps, ROUND(AVG(e.salary), 0) AS avg_salary
FROM employees e
JOIN departments d ON d.dept_id = e.dept_id
GROUP BY d.dept_name
HAVING COUNT(*) >= 2
ORDER BY avg_salary DESC
''')

,dept_name,n_emps,avg_salary
0,Engineering,2,152500.0
1,Data Science,3,133333.0
2,Sales,3,85000.0


**The line that scores:** *"WHERE filters rows before grouping; HAVING filters groups after aggregation — you can't use an aggregate in WHERE."* Also note the `JOIN` happens conceptually before `GROUP BY`, which is why the count reflects joined rows only (Ivan, with `dept_id NULL`, drops out of an inner join — say that unprompted).

## Q3 — LEFT JOIN + IS NULL: the anti-join

*Which departments have no employees at all?* This is the classic that tests whether someone actually understands outer joins — `Research` (id 5) has nobody.

In [4]:
q('''
SELECT d.dept_name
FROM departments d
LEFT JOIN employees e ON e.dept_id = d.dept_id
WHERE e.emp_id IS NULL
''')

,dept_name
0,Research


## Q4 — Self-Join: employee and their manager in one row

In [5]:
q('''
SELECT e.name AS employee, m.name AS manager, e.salary
FROM employees e
LEFT JOIN employees m ON e.manager_id = m.emp_id
ORDER BY e.emp_id
LIMIT 6
''')

,employee,manager,salary
0,Ada,NaN,185000
1,Bruno,Ada,120000
2,Chen,Ada,140000
3,Disha,Chen,132000
4,Emil,Chen,128000
5,Farah,Ada,95000


**Points:** self-join needs **aliases** (mandatory, not stylistic); LEFT JOIN so the CEO (Ada, `manager_id NULL`) still appears with a NULL manager. In pandas this is `df.merge(df, left_on="manager_id", right_on="emp_id")` — mentioning the translation shows range.

## Q5 — Subquery vs CTE vs Correlated

*Average salary by department, only for departments above the company-wide average.* Three syntaxes, same answer — show you can pick for readability:

In [6]:
# subquery version
print(q('''
SELECT dept_id, AVG(salary) AS dept_avg
FROM employees
GROUP BY dept_id
HAVING AVG(salary) > (SELECT AVG(salary) FROM employees)
'''))

# CTE version — the one to write in interviews (readable, named stages)
print(q('''
WITH company AS (SELECT AVG(salary) AS overall FROM employees)
SELECT dept_id, AVG(salary) AS dept_avg
FROM employees
GROUP BY dept_id
HAVING AVG(salary) > (SELECT overall FROM company)
'''))

   dept_id       dept_avg
0        1  152500.000000
1        2  133333.333333
   dept_id       dept_avg
0        1  152500.000000
1        2  133333.333333


**Correlated subquery** — the *for each row* pattern (runs the inner query per outer row; fine for interviews, flag the cost in production):

In [7]:
q('''
SELECT e.name, e.salary, e.dept_id,
       (SELECT ROUND(AVG(e2.salary), 0)
        FROM employees e2
        WHERE e2.dept_id = e.dept_id) AS dept_avg
FROM employees e
WHERE e.salary > (SELECT AVG(e2.salary)
                  FROM employees e2
                  WHERE e2.dept_id = e.dept_id)
ORDER BY e.dept_id, e.salary DESC
''')

,name,salary,dept_id,dept_avg
0,Ada,185000,1,152500.0
1,Chen,140000,2,133333.0
2,Farah,95000,3,85000.0


## Q6 — NULL Logic: COALESCE, NULLIF, IS NULL

In [8]:
q('''
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN dept_id IS NULL THEN 1 ELSE 0 END) AS no_dept,
  COUNT(dept_id)  AS non_null_depts,   -- COUNT skips NULLs
  COUNT(DISTINCT dept_id) AS distinct_depts,
  COALESCE(NULLIF(manager_id, 1), 0) AS manager_or_zero
FROM employees
''')

,total_rows,no_dept,non_null_depts,distinct_depts,manager_or_zero
0,10,1,9,4,0


**The three NULL rules to recite:** (1) `COUNT(col)` skips NULLs but `COUNT(*)` doesn't; (2) any comparison with NULL yields NULL (so `salary > NULL` filters the row *out*, silently); (3) `COALESCE` for defaults, `NULLIF(a, b)` for "treat b as missing" — e.g. avoiding divide-by-zero: `x / NULLIF(y, 0)`.

## Q7 — CASE Pivot: order status counts as columns

In [9]:
q('''
SELECT e.dept_id,
       SUM(CASE WHEN o.status = 'shipped'   THEN 1 ELSE 0 END) AS shipped,
       SUM(CASE WHEN o.status = 'returned'  THEN 1 ELSE 0 END) AS returned,
       SUM(CASE WHEN o.status = 'cancelled' THEN 1 ELSE 0 END) AS cancelled,
       SUM(CASE WHEN o.status = 'pending'   THEN 1 ELSE 0 END) AS pending,
       ROUND(SUM(o.amount), 2) AS total_amount
FROM orders o
JOIN employees e ON e.emp_id = o.emp_id
GROUP BY e.dept_id
ORDER BY total_amount DESC
''')

,dept_id,shipped,returned,cancelled,pending,total_amount
0,2.0,4,1,0,0,2285.50
1,3.0,2,0,1,0,1495.25
2,4.0,0,0,0,1,220.00
3,NaN,1,0,0,0,99.99


Conditional aggregation — the standard SQL pivot. It's also the pattern behind leakage-safe feature building ("per-user counts of each event type") — say that connection and you sound like an ML engineer, not just a query writer.

## Q8 — Top-N per Group (the prelude to window functions)

*Highest-paid employee per department.* The window-function version lands in the next notebook; the classic pre-window version uses a correlated subquery — know both:

In [10]:
q('''
SELECT e.dept_id, e.name, e.salary
FROM employees e
WHERE e.salary = (SELECT MAX(e2.salary)
                  FROM employees e2
                  WHERE e2.dept_id = e.dept_id)
ORDER BY e.dept_id
''')
print("note: ties both appear — MAX-matching keeps ties, ROW_NUMBER() would not")

note: ties both appear — MAX-matching keeps ties, ROW_NUMBER() would not


## Q9 — Dates: month extraction and quarter grouping

In [11]:
q('''
SELECT substr(placed_at, 1, 7) AS month,
       COUNT(*) AS n_orders,
       ROUND(SUM(amount), 2) AS revenue,
       ROUND(AVG(amount), 2) AS avg_order
FROM orders
GROUP BY substr(placed_at, 1, 7)
ORDER BY month
''')

,month,n_orders,revenue,avg_order
0,2026-01,2,570.50,285.25
1,2026-02,2,1200.00,600.00
2,2026-03,4,1450.25,362.56
3,2026-04,2,879.99,440.00


SQLite stores dates as TEXT (ISO-8601 sorts correctly — *that's the reason for the format*, a good point to make); PostgreSQL/MySQL have real date types and `DATE_TRUNC`/`strftime`. Say which dialect you're in before writing date SQL.

## Q10 — The Business Question Round

Interviewers escalate from syntax to *thinking*. Rehearse answering these against this schema out loud:

1. *Revenue per employee, only counting shipped orders, showing zero for employees with none* → LEFT JOIN + CASE-when-shipped sum, or `SUM(CASE WHEN status='shipped' THEN amount END)` which naturally yields NULL→COALESCE to 0.
2. *Month-over-month revenue growth* → this month vs previous month: window `LAG` (next notebook) or a self-join on `month = month+1`.
3. *Employees earning more than their manager* → self-join + comparison.
4. *Departments where average salary exceeds the average of averages* → nested aggregation, group-then-average.
5. *Retention: % of January customers who ordered again in February* → two filtered aggregates over a customers CTE; say — this is where SQL starts feeling like analytics.

In [12]:
# answer to #1 and #3, executed:
print(q('''
SELECT e.name,
       COALESCE(SUM(CASE WHEN o.status = 'shipped' THEN o.amount END), 0) AS shipped_revenue
FROM employees e
LEFT JOIN orders o ON o.emp_id = e.emp_id
GROUP BY e.name
ORDER BY shipped_revenue DESC
LIMIT 5
'''))
print(q('''
SELECT e.name, e.salary
FROM employees e
JOIN employees m ON e.manager_id = m.emp_id
WHERE e.salary > m.salary
'''))

     name  shipped_revenue
0    Chen          1085.50
1    Jaya           780.00
2  Gustav           640.00
3   Disha           310.00
4    Ivan            99.99
Empty DataFrame
Columns: [name, salary]
Index: []


## Rapid-Fire: SQL Vocabulary They Expect

| Question | The 15-second answer |
| --- | --- |
| WHERE vs HAVING? | Row filter before grouping vs group filter after aggregation |
| Inner vs left vs full vs cross join? | Matched only; all left + matched; all of both; every pairing (n×m) — in that order |
| UNION vs UNION ALL? | UNION dedupes (sort/hash cost); UNION ALL keeps duplicates — faster when dups impossible/acceptable |
| DELETE vs TRUNCATE? | DELETE logs rows, can WHERE, rollback-able; TRUNCATE drops/recreates pages — fast, no WHERE, DDL-ish |
| What's an index tradeoff? | Read speed (B-tree lookup) vs write cost + storage; every index slows INSERT/UPDATE — index for the queries you actually run |
| Primary vs unique key? | PK: identity, one per table, no NULLs; UNIQUE: no dups but NULLs allowed (multiple, in most engines) |
| Transaction ACID? | Atomicity (all-or-nothing), Consistency (constraints hold), Isolation (concurrent correctness), Durability (committed survives crash) |
| OLTP vs OLAP? | Row-store, many small writes, normalized vs column-store, bulk reads, denormalized star schemas — warehouses are OLAP |
| EXPLAIN / query plan? | Ask the engine how it will run the query — scan vs index seek, join order; the first tool when a query is slow |
| Window functions? | Aggregate over a sliding frame without collapsing rows — the subject of the next notebook |